# TASK 3 — Data Cleaning

## Project: Cafe Sales Data Cleaning

### Objective
The objective of this project is to demonstrate professional-level data
cleaning skills by identifying and resolving missing values, duplicate
records, inconsistent categorical values, incorrect data types, invalid
values, and outliers.

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("dirty_cafe_sales.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.nunique()

In [ ]:
df.isnull().sum()

In [ ]:
missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

missing_report

In [ ]:
for column in df.columns:
    print("\n", column)
    print(df[column].value_counts(dropna=False).head(15))

In [ ]:
quality_report_before = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing %": (df.isnull().mean() * 100).round(2).values,
    "Unique Values": df.nunique().values
})

quality_report_before

In [ ]:
before_rows = len(df)

before_duplicates = df.duplicated().sum()

before_missing = df.isnull().sum().sum()

print("Rows before cleaning:", before_rows)
print("Duplicates before cleaning:", before_duplicates)
print("Missing values before cleaning:", before_missing)

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Duplicates after removal:", df.duplicated().sum())

### Duplicate Removal

Duplicate rows were checked using `duplicated()`.

No exact duplicate records were found in the original dataset, so no rows
were removed during duplicate cleaning.

In [ ]:
df = df.replace(["ERROR", "UNKNOWN"], np.nan)

In [ ]:
df.isnull().sum()

## Standardization of Categorical Data

Categorical columns are standardized by removing unnecessary whitespace and
using consistent capitalization.

In [ ]:
df["Item"] = df["Item"].str.strip()
df["Payment Method"] = df["Payment Method"].str.strip()
df["Location"] = df["Location"].str.strip()

df["Item"] = df["Item"].str.title()
df["Payment Method"] = df["Payment Method"].str.title()
df["Location"] = df["Location"].str.title()

df["Location"] = df["Location"].replace({
    "In-Store": "In-store"
})

In [ ]:
print("Items:")
print(df["Item"].unique())

print("\nPayment Methods:")
print(df["Payment Method"].unique())

print("\nLocations:")
print(df["Location"].unique())

In [ ]:
df["Item"].value_counts(dropna=False)

In [ ]:
df["Item"] = df["Item"].fillna(df["Item"].mode()[0])

In [ ]:
df["Item"].isnull().sum()

### Item Missing Values

The Item column is categorical. Missing values were replaced using the mode,
because the most frequently occurring valid item provides a reasonable
representative category while preserving the number of records.

In [ ]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

In [ ]:
df["Quantity"].isnull().sum()

In [ ]:
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

In [ ]:
df["Quantity"] = df["Quantity"].astype(int)

In [ ]:
print(df["Quantity"].dtype)
print(df["Quantity"].isnull().sum())

### Quantity Missing Values

Quantity is a numerical variable. Median imputation was selected because
median is less affected by extreme values than the mean and is appropriate
for transaction quantities.

In [ ]:
df["Price Per Unit"] = pd.to_numeric(
    df["Price Per Unit"],
    errors="coerce"
)

In [ ]:
df["Price Per Unit"].isnull().sum()

In [ ]:
df["Price Per Unit"] = df["Price Per Unit"].fillna(
    df["Price Per Unit"].median()
)

In [ ]:
df["Price Per Unit"].isnull().sum()

### Price Per Unit Missing Values

Price Per Unit is numerical, so invalid and missing values were converted to
NaN and replaced using median imputation. Median was selected because it is
more robust to potential outliers than mean imputation.

In [ ]:
df["Total Spent"] = pd.to_numeric(
    df["Total Spent"],
    errors="coerce"
)

In [ ]:
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

In [ ]:
df[["Quantity", "Price Per Unit", "Total Spent"]].head()

In [ ]:
df["Payment Method"].value_counts(dropna=False)

In [ ]:
df["Payment Method"] = df["Payment Method"].fillna(
    df["Payment Method"].mode()[0]
)

In [ ]:
df["Payment Method"].isnull().sum()

### Payment Method Missing Values

Payment Method is a categorical variable. Missing and invalid values were
replaced with the mode because the most common valid payment method is a
reasonable replacement for missing categorical observations.

In [ ]:
df["Location"].value_counts(dropna=False)

In [ ]:
df["Location"] = df["Location"].fillna(
    df["Location"].mode()[0]
)

In [ ]:
df["Location"].isnull().sum()

### Location Missing Values

Location is categorical, so missing values were replaced using the mode.
This avoids deleting valid transaction records unnecessarily.

In [ ]:
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [ ]:
df["Transaction Date"].isnull().sum()

In [ ]:
before_date_drop = len(df)

df = df.dropna(subset=["Transaction Date"])

after_date_drop = len(df)

print(
    "Rows removed because transaction date was unavailable:",
    before_date_drop - after_date_drop
)

### Transaction Date Cleaning

Transaction Date was converted to datetime using `pd.to_datetime()`.

Invalid dates were converted to NaT. Since the transaction date is an
important identifier for time-based analysis, records without a valid
transaction date were removed instead of assigning an artificial date.

In [ ]:
df["Transaction ID"] = df["Transaction ID"].astype(str)

In [ ]:
df["Transaction ID"].dtype

### Transaction ID

Transaction ID was kept as a string because it is an identifier rather than
a numerical measurement. Performing mathematical operations on transaction
IDs would not be meaningful.

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
total_missing_after = df.isnull().sum().sum()

print("Total missing values after cleaning:", total_missing_after)

In [ ]:
print("Items:")
print(df["Item"].unique())

print("\nPayment Methods:")
print(df["Payment Method"].unique())

print("\nLocations:")
print(df["Location"].unique())

In [ ]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]
    
    return outliers, lower_bound, upper_bound

In [ ]:
outliers_quantity, lower_q, upper_q = detect_outliers_iqr(
    df,
    "Quantity"
)

print("Quantity lower bound:", lower_q)
print("Quantity upper bound:", upper_q)
print("Quantity outliers:", len(outliers_quantity))

In [ ]:
outliers_price, lower_p, upper_p = detect_outliers_iqr(
    df,
    "Price Per Unit"
)

print("Price lower bound:", lower_p)
print("Price upper bound:", upper_p)
print("Price outliers:", len(outliers_price))

In [ ]:
outliers_total, lower_t, upper_t = detect_outliers_iqr(
    df,
    "Total Spent"
)

print("Total Spent lower bound:", lower_t)
print("Total Spent upper bound:", upper_t)
print("Total Spent outliers:", len(outliers_total))

In [ ]:
outlier_summary = pd.DataFrame({
    "Column": [
        "Quantity",
        "Price Per Unit",
        "Total Spent"
    ],
    "Outliers": [
        len(outliers_quantity),
        len(outliers_price),
        len(outliers_total)
    ]
})

outlier_summary

In [ ]:
df[["Quantity", "Price Per Unit", "Total Spent"]].describe()

In [ ]:
def cap_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    data[column] = data[column].clip(
        lower_bound,
        upper_bound
    )

    return data


df = cap_outliers_iqr(df, "Quantity")
df = cap_outliers_iqr(df, "Price Per Unit")

df["Total Spent"] = (
    df["Quantity"] * df["Price Per Unit"]
)

## Outlier Treatment

Outliers were detected using the Interquartile Range (IQR) method.

Instead of deleting potentially valid transaction records, extreme values
in Quantity and Price Per Unit were capped at their IQR-based lower and
upper bounds.

Total Spent was not capped separately because it is a derived variable.
Instead, it was recalculated after treating Quantity and Price Per Unit
outliers.

Total Spent = Quantity × Price Per Unit

This approach reduces the influence of extreme values while retaining
transaction records.

In [ ]:
calculated_total = df["Quantity"] * df["Price Per Unit"]

print(
    "Incorrect Total Spent values:",
    (df["Total Spent"] != calculated_total).sum()
)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print(
    "Duplicate Transaction IDs:",
    df["Transaction ID"].duplicated().sum()
)

In [ ]:
quality_report_after = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing %": (df.isnull().mean() * 100).round(2).values,
    "Unique Values": df.nunique().values
})

quality_report_after

In [ ]:
after_rows = len(df)

after_duplicates = df.duplicated().sum()

after_missing = df.isnull().sum().sum()

In [ ]:
before_after = pd.DataFrame({
    "Metric": [
        "Row Count",
        "Duplicate Rows",
        "Missing Values"
    ],
    "Before Cleaning": [
        before_rows,
        before_duplicates,
        before_missing
    ],
    "After Cleaning": [
        after_rows,
        after_duplicates,
        after_missing
    ]
})

before_after

In [ ]:
datatype_summary = pd.DataFrame({
    "Column": df.columns,
    "Final Data Type": df.dtypes.astype(str).values
})

datatype_summary

In [ ]:
df.head(10)

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

# Export Cleaned Dataset

The cleaned dataset is exported as a new CSV file for further analysis and
visualization.

In [ ]:
df.to_csv(
    "cleaned_cafe_sales.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

In [ ]:
cleaned_df = pd.read_csv("cleaned_cafe_sales.csv")

print("Cleaned dataset shape:", cleaned_df.shape)

cleaned_df.head()

In [ ]:
print("Missing values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate rows:")
print(cleaned_df.duplicated().sum())

# Conclusion

The Cafe Sales dataset was successfully cleaned and transformed into an
analysis-ready dataset.

The cleaning process included:

- Identifying missing values
- Handling ERROR and UNKNOWN values
- Checking and removing duplicate records
- Standardizing categorical data
- Handling missing numerical values
- Handling missing categorical values
- Converting columns to appropriate data types
- Converting Transaction Date to datetime
- Keeping Transaction ID as a string
- Recalculating Total Spent using Quantity × Price Per Unit
- Detecting outliers using the IQR method
- Treating numerical outliers
- Validating the cleaned dataset
- Comparing the dataset before and after cleaning
- Exporting the final cleaned dataset as a CSV file

The final dataset is clean, consistent, validated, and ready for further
analysis and visualization.